In [1]:
from pathlib import Path
import json
from openpyxl import load_workbook
from copy import copy

BASE = Path("/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent")

TEMPLATE_XLSX = BASE / "rubrics/Video_Selection_Rubric_AutoScore.xlsx"
TRANSCRIPTS_ROOT = BASE / "transcripts"
OUT_XLSX = BASE / "rubrics/Video_Selection_Rubric_Rebuilt_WITH_FORMULAS.xlsx"

# 1) Discover transcripts recursively
txt_files = sorted(TRANSCRIPTS_ROOT.rglob("*.txt"))
print("Discovered .txt transcripts:", len(txt_files))

# 2) Load template workbook (this contains your formulas)
wb = load_workbook(TEMPLATE_XLSX)
ws = wb["Video_Selection_Rubric"] if "Video_Selection_Rubric" in wb.sheetnames else wb.active

# 3) Build header -> column index map
headers = {}
for c in range(1, ws.max_column + 1):
    v = ws.cell(row=1, column=c).value
    if isinstance(v, str) and v.strip():
        headers[v.strip()] = c

def col(name):
    if name not in headers:
        raise KeyError(f"Template missing required column: {name}")
    return headers[name]

# Required columns (must exist in your template)
COL_VIDEO_ID = col("Video_ID")

# Optional metadata columns: only fill if they exist in your template
optional_cols = {
    "Channel_ID": headers.get("Channel_ID"),
    "Channel_Name": headers.get("Channel_Name"),
    "Video_Title": headers.get("Video_Title"),
    "YouTube_Video_ID": headers.get("YouTube_Video_ID"),
    "PublishedAt": headers.get("PublishedAt"),
    "Video_Length_Minutes": headers.get("Video_Length_Minutes"),
    "Transcript_Available (Yes/No)": headers.get("Transcript_Available (Yes/No)"),
}

# 4) Identify the “template row” that contains formulas
# In your rubric, row 2 is guidance; row 3 is usually the first data row with formulas.
TEMPLATE_ROW = 3

def copy_row_style_and_formulas(src_row: int, dst_row: int):
    """Copy cell value (including formulas), style, and formatting from src_row to dst_row."""
    for c in range(1, ws.max_column + 1):
        src = ws.cell(row=src_row, column=c)
        dst = ws.cell(row=dst_row, column=c)

        dst.value = src.value  # this copies formulas too

        if src.has_style:
            dst._style = copy(src._style)
        dst.number_format = src.number_format
        dst.font = copy(src.font)
        dst.border = copy(src.border)
        dst.fill = copy(src.fill)
        dst.alignment = copy(src.alignment)
        dst.protection = copy(src.protection)
        dst.comment = None

# 5) Clear existing data rows (optional but recommended for clean rebuild)
# Keep header row (1) and guidance row (2), keep template row (3) as the row we copy from.
# We'll delete everything below row 3 and then rebuild.
if ws.max_row > TEMPLATE_ROW:
    ws.delete_rows(TEMPLATE_ROW + 1, ws.max_row - TEMPLATE_ROW)

# 6) Build rows for every transcript, preserving formulas by cloning TEMPLATE_ROW
start_write_row = TEMPLATE_ROW
for i, txt_path in enumerate(txt_files):
    vid = txt_path.stem
    meta_path = txt_path.with_suffix(".meta.json")
    meta = {}
    if meta_path.exists():
        meta = json.loads(meta_path.read_text(encoding="utf-8"))

    dst_row = start_write_row + i

    # Make sure the row exists and has formulas/styles
    if dst_row == TEMPLATE_ROW:
        # Reuse the existing template row for the first entry
        pass
    else:
        ws.insert_rows(dst_row)
        copy_row_style_and_formulas(TEMPLATE_ROW, dst_row)

    # Fill required + optional fields (only overwrite these “input” cells)
    ws.cell(row=dst_row, column=COL_VIDEO_ID, value=vid)

    if optional_cols["Channel_ID"]:
        ws.cell(row=dst_row, column=optional_cols["Channel_ID"], value=meta.get("Channel_ID",""))
    if optional_cols["Channel_Name"]:
        ws.cell(row=dst_row, column=optional_cols["Channel_Name"], value=meta.get("Channel_Name",""))
    if optional_cols["Video_Title"]:
        ws.cell(row=dst_row, column=optional_cols["Video_Title"], value=meta.get("Video_Title",""))
    if optional_cols["YouTube_Video_ID"]:
        ws.cell(row=dst_row, column=optional_cols["YouTube_Video_ID"], value=meta.get("YouTube_Video_ID",""))
    if optional_cols["PublishedAt"]:
        ws.cell(row=dst_row, column=optional_cols["PublishedAt"], value=meta.get("PublishedAt",""))

    if optional_cols["Video_Length_Minutes"]:
        dur = meta.get("DurationSeconds", "")
        try:
            mins = round(float(dur)/60.0, 2) if dur != "" else ""
        except Exception:
            mins = ""
        ws.cell(row=dst_row, column=optional_cols["Video_Length_Minutes"], value=mins)

    if optional_cols["Transcript_Available (Yes/No)"]:
        ws.cell(row=dst_row, column=optional_cols["Transcript_Available (Yes/No)"], value="Yes")

# 7) Save
wb.save(OUT_XLSX)
print("Saved rebuilt rubric WITH formulas to:", OUT_XLSX)

Discovered .txt transcripts: 440
Saved rebuilt rubric WITH formulas to: /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/rubrics/Video_Selection_Rubric_Rebuilt_WITH_FORMULAS.xlsx


In [3]:
from pathlib import Path
import re, json
from typing import Dict, List, Optional
import pandas as pd
from openpyxl import load_workbook

# -----------------------------
# 1) Keyword rules
# -----------------------------
TTP_KEYWORDS = [
    r"\bphish\w*\b", r"\bphishing\b", r"\bmalspam\b", r"\bvishing\b",
    r"\brdp\b", r"\bbrute\s+force\b", r"\bpassword\s+spray\b",
    r"\bexploit\w*\b", r"\bvulnerab\w*\b", r"\b0-day\b",
    r"\bcobalt\s+strike\b", r"\bbeacon\b", r"\bmimikatz\b",
    r"\bpsexec\b", r"\bwmi\b", r"\bwinrm\b",
    r"\brclone\b", r"\bmeganz\b", r"\bwinscp\b",
    r"\bexfil\w*\b", r"\bdata\s+theft\b", r"\bdouble\s+extortion\b", r"\bleak\s+site\b",
    r"\bshadow\s+copy\b", r"\bvss\b", r"\bransom\s+note\b", r"\bencrypt\w*\b",
    r"\bscheduled\s+task\b", r"\bgpo\b", r"\bgroup\s+policy\b",
]

DFIR_CASE_STUDY_SIGNALS = [
    r"\bcase\s+summary\b", r"\bintrusion\b", r"\bwe\s+observed\b", r"\btimeline\b",
    r"\bbeach\s*head\b", r"\bday\s+\d+\b", r"\bincident\s+response\b", r"\bforensic\w*\b",
]

PLATFORM_PATTERNS = {
    "ESXi": [r"\besxi\b", r"\bvmware\b", r"\bvcenter\b"],
    "Linux": [r"\blinux\b", r"\bubuntu\b", r"\bdebian\b", r"\bcentos\b", r"\brhel\b", r"\bred\s*hat\b"],
    "Windows": [r"\bwindows\b", r"\bactive\s+directory\b", r"\bdomain\s+controller\b", r"\blsass\b"],
}

DEPTH_HIGH = [
    r"\batt&ck\b", r"\bmitre\b", r"\btelemetry\b", r"\bioc(s)?\b", r"\byara\b", r"\bsigma\b",
    r"\bpcap\b", r"\bmemory\s+dump\b", r"\breverse\s+engineering\b", r"\bprocess\s+injection\b",
    r"\blog(s)?\b", r"\bevidence\b", r"\bdetection\s+rule\b",
]
DEPTH_MED = [
    r"\bkill\s+chain\b", r"\blateral\s+movement\b", r"\bprivilege\s+escalation\b",
    r"\bexfiltration\b", r"\bpersistence\b", r"\bdiscovery\b",
]

RED_FLAG_PATTERNS = {
    "Marketing": [
        r"\b(contact\s+us|pricing|book\s+a\s+demo|request\s+a\s+demo)\b",
        r"\bour\s+product\b", r"\bwebinar\b", r"\bsponsored\b",
    ],
    "News": [
        r"\bbreaking\s+news\b", r"\bheadlines\b", r"\bnews\s+update\b",
    ],
    "LowSignal": [
        r"\b(what\s+is\s+ransomware|ransomware\s+explained)\b"
    ]
}

def normalize(text: str) -> str:
    return re.sub(r"\s+", " ", text.lower())

def any_hit(text: str, patterns: List[str]) -> bool:
    return any(re.search(p, text, flags=re.IGNORECASE) for p in patterns)

def count_hits(text: str, patterns: List[str]) -> int:
    return sum(len(re.findall(p, text, flags=re.IGNORECASE)) for p in patterns)

def detect_platform(text: str) -> str:
    hits = {plat: count_hits(text, pats) for plat, pats in PLATFORM_PATTERNS.items()}
    chosen = [p for p, h in hits.items() if h > 0]
    if not chosen:
        return ""
    if len(chosen) == 1:
        return chosen[0]
    return "Mixed"

def detect_depth(text: str) -> str:
    high = count_hits(text, DEPTH_HIGH)
    med = count_hits(text, DEPTH_MED)
    if high >= 6:
        return "High"
    if high >= 2 or med >= 4:
        return "Medium"
    return "Low"

def detect_red_flags(text: str) -> str:
    flags = [label for label, pats in RED_FLAG_PATTERNS.items() if any_hit(text, pats)]
    return "/".join(flags)

# -----------------------------
# 2) Family alias list loader
# -----------------------------
def load_family_aliases_xlsx(path: str) -> Dict[str, str]:
    df = pd.read_excel(path).fillna("")
    alias_map: Dict[str, str] = {}
    for _, row in df.iterrows():
        fam = str(row.get("Ransomware_Family_Name","")).strip()
        if not fam:
            continue
        alias_map[fam.lower()] = fam
        aliases = str(row.get("Alias_Names","")).strip()
        if aliases and aliases != "—":
            for a in [x.strip() for x in aliases.split(",")]:
                if a:
                    alias_map[a.lower()] = fam
    return alias_map

def detect_specific_family(text: str, alias_map: Optional[Dict[str,str]]) -> bool:
    if not alias_map:
        return False
    # Fast check: look for any alias as a whole word
    for alias in alias_map.keys():
        alias_re = re.escape(alias).replace(r"\ ", r"\s+")
        if re.search(rf"\b{alias_re}\b", text, flags=re.IGNORECASE):
            return True
    return False

# -----------------------------
# 3) Python scoring (no Excel dependence)
# -----------------------------
def yn(val) -> bool:
    return str(val).strip().lower() == "yes"

def depth_score(x):
    x = str(x).strip().lower()
    return 2 if x == "high" else 1 if x == "medium" else 0

def recency_bucket(published_at: str) -> str:
    # simple bucket from year
    s = str(published_at).strip()
    if not s:
        return ""
    m = re.search(r"(\d{4})", s)
    if not m:
        return ""
    y = int(m.group(1))
    if y < 2020: return "Pre-2020"
    if 2020 <= y <= 2022: return "2020-2022"
    return "2023+"

def recency_score(x):
    x = str(x).strip()
    return 1 if x == "2023+" else 0.5 if x == "2020-2022" else 0

def platform_score(x):
    return 1 if str(x).strip() != "" else 0

def redflag_penalty(x):
    return -2 if str(x).strip() != "" else 0

# -----------------------------
# 4) Main function: recursive scan + fill + score
# -----------------------------
def fill_rubric_recursive_with_py_scores(
    rubric_xlsx: str,
    transcripts_dir: str,
    out_xlsx: str,
    alias_map: Optional[Dict[str,str]] = None
):
    transcripts_root = Path(transcripts_dir)

    # Index all transcripts recursively
    txt_files = sorted(transcripts_root.rglob("*.txt"))
    txt_index = {p.stem: p for p in txt_files}

    # meta: V0001.meta.json
    meta_index = {}
    for p in transcripts_root.rglob("*.meta.json"):
        vid = p.name.replace(".meta.json", "")
        meta_index[vid] = p

    print("Transcripts found:", len(txt_index))

    # Load rubric template workbook
    wb = load_workbook(rubric_xlsx)
    ws = wb["Video_Selection_Rubric"] if "Video_Selection_Rubric" in wb.sheetnames else wb.active

    # Header mapping
    headers = {}
    for c in range(1, ws.max_column + 1):
        v = ws.cell(row=1, column=c).value
        if isinstance(v, str) and v.strip():
            headers[v.strip()] = c

    def h(name: str) -> int:
        if name not in headers:
            raise KeyError(f"Missing column in rubric: {name}")
        return headers[name]

    # Required rubric input columns
    COL_VID = h("Video_ID")
    COL_FAM_MENTION = h("Ransomware_Family_Mentioned (Yes/No)")
    COL_SPEC_FAM = h("Specific_Family_Named (Yes/No)")
    COL_TTPS = h("TTPs_Mentioned (Yes/No)")
    COL_DFIR = h("DFIR_Case_Study (Yes/No)")
    COL_PLATFORM = h("Platform_Mentioned (Windows/Linux/ESXi)")
    COL_TRANSCRIPT = h("Transcript_Available (Yes/No)")
    COL_DEPTH = h("Estimated_Technical_Depth (Low/Medium/High)")
    COL_REDFLAG = h("Red_Flags (News/Marketing/LowSignal)")
    COL_RECENCY = h("Recency (Pre-2020 / 2020-2022 / 2023+)")

    # Optional meta columns (fill if present)
    COL_CH_ID = headers.get("Channel_ID", None)
    COL_CH_NAME = headers.get("Channel_Name", None)
    COL_TITLE = headers.get("Video_Title", None)
    COL_YT_ID = headers.get("YouTube_Video_ID", None)
    COL_PUB = headers.get("PublishedAt", None)
    COL_MIN = headers.get("Video_Length_Minutes", None)

    # Add Python-computed score columns if missing
    if "Selection_Score_PY" not in headers:
        ws.cell(row=1, column=ws.max_column + 1, value="Selection_Score_PY")
        ws.cell(row=1, column=ws.max_column + 1, value="Include_PY")
        # rebuild headers
        headers = {ws.cell(row=1, column=c).value: c for c in range(1, ws.max_column + 1) if ws.cell(row=1, column=c).value}
    COL_SCORE_PY = headers["Selection_Score_PY"]
    COL_INCLUDE_PY = headers["Include_PY"]

    def set_cell(r, c, val):
        ws.cell(row=r, column=c, value=val)

    rows_processed = 0
    rows_missing = 0

    for r in range(3, ws.max_row + 1):
        vid = ws.cell(row=r, column=COL_VID).value
        if not vid:
            continue
        vid = str(vid).strip()
        if not vid.startswith("V"):
            continue

        tfile = txt_index.get(vid)
        if not tfile:
            set_cell(r, COL_TRANSCRIPT, "No")
            rows_missing += 1
            continue

        # Read transcript + meta
        text_raw = tfile.read_text(encoding="utf-8", errors="ignore")
        text = normalize(text_raw)

        meta = {}
        mp = meta_index.get(vid)
        if mp and mp.exists():
            meta = json.loads(mp.read_text(encoding="utf-8"))

        # Fill meta fields if the columns exist
        if COL_CH_ID:   set_cell(r, COL_CH_ID, meta.get("Channel_ID",""))
        if COL_CH_NAME: set_cell(r, COL_CH_NAME, meta.get("Channel_Name",""))
        if COL_TITLE:   set_cell(r, COL_TITLE, meta.get("Video_Title",""))
        if COL_YT_ID:   set_cell(r, COL_YT_ID, meta.get("YouTube_Video_ID",""))
        if COL_PUB:     set_cell(r, COL_PUB, meta.get("PublishedAt",""))
        if COL_MIN:
            dur = meta.get("DurationSeconds","")
            try:
                mins = round(float(dur)/60.0, 2) if dur != "" else ""
            except Exception:
                mins = ""
            set_cell(r, COL_MIN, mins)

        # Rubric fields
        set_cell(r, COL_TRANSCRIPT, "Yes")
        set_cell(r, COL_FAM_MENTION, "Yes" if re.search(r"\bransomware\b", text, re.I) else "No")
        set_cell(r, COL_SPEC_FAM, "Yes" if detect_specific_family(text, alias_map) else "No")
        set_cell(r, COL_TTPS, "Yes" if any_hit(text, TTP_KEYWORDS) else "No")
        set_cell(r, COL_DFIR, "Yes" if any_hit(text, DFIR_CASE_STUDY_SIGNALS) else "No")

        platform = detect_platform(text)
        if platform:
            set_cell(r, COL_PLATFORM, platform)

        depth = detect_depth(text)
        set_cell(r, COL_DEPTH, depth)

        flags = detect_red_flags(text)
        if flags:
            set_cell(r, COL_REDFLAG, flags)
        else:
            set_cell(r, COL_REDFLAG, "")

        # Recency bucket from PublishedAt meta
        bucket = recency_bucket(meta.get("PublishedAt",""))
        if bucket:
            set_cell(r, COL_RECENCY, bucket)

        # Python score (no Excel calc needed)
        score_py = (
            (1 if yn(ws.cell(r, COL_FAM_MENTION).value) else 0) +
            (2 if yn(ws.cell(r, COL_SPEC_FAM).value) else 0) +
            (2 if yn(ws.cell(r, COL_TTPS).value) else 0) +
            (2 if yn(ws.cell(r, COL_DFIR).value) else 0) +
            platform_score(ws.cell(r, COL_PLATFORM).value) +
            (2 if yn(ws.cell(r, COL_TRANSCRIPT).value) else 0) +
            depth_score(ws.cell(r, COL_DEPTH).value) +
            recency_score(ws.cell(r, COL_RECENCY).value) +
            redflag_penalty(ws.cell(r, COL_REDFLAG).value)
        )

        set_cell(r, COL_SCORE_PY, score_py)
        set_cell(r, COL_INCLUDE_PY, "Yes" if score_py >= 6 else "No")

        rows_processed += 1

    wb.save(out_xlsx)
    print("Rows processed:", rows_processed)
    print("Rows missing transcript:", rows_missing)
    print("Saved to:", out_xlsx)

# -----------------------------
# 5) RUN
# -----------------------------
alias_map = load_family_aliases_xlsx(
    "/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/rubrics/Ransomware_Family_Coverage_List.xlsx"
)

fill_rubric_recursive_with_py_scores(
    rubric_xlsx="/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/rubrics/Video_Selection_Rubric_Rebuilt_WITH_FORMULAS.xlsx",
    transcripts_dir="/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts",
    out_xlsx="/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/outputs/Video_Selection_Rubric_AutoScore_Filled.xlsx",
    alias_map=alias_map
)

Transcripts found: 440
Rows processed: 439
Rows missing transcript: 0
Saved to: /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/outputs/Video_Selection_Rubric_AutoScore_Filled.xlsx


In [4]:
from pathlib import Path
import json, re
import pandas as pd
from openpyxl import load_workbook

# -----------------------
# Paths
# -----------------------
BASE = Path("/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent")
RUBRIC_IN = BASE / "rubrics/Video_Selection_Rubric_Rebuilt_WITH_FORMULAS.xlsx"
TRANSCRIPTS_ROOT = BASE / "transcripts"
FAMILY_LIST = BASE / "rubrics/Ransomware_Family_Coverage_List.xlsx"
RUBRIC_OUT = BASE / "outputs/Video_Selection_Rubric_Filled_Columns_ONLY.xlsx"

# -----------------------
# Keyword rules (yours)
# -----------------------
TTP_KEYWORDS = [
    r"\bphish\w*\b", r"\bphishing\b", r"\bmalspam\b", r"\bvishing\b",
    r"\brdp\b", r"\bbrute\s+force\b", r"\bpassword\s+spray\b",
    r"\bexploit\w*\b", r"\bvulnerab\w*\b", r"\b0-day\b",
    r"\bcobalt\s+strike\b", r"\bbeacon\b", r"\bmimikatz\b",
    r"\bpsexec\b", r"\bwmi\b", r"\bwinrm\b",
    r"\brclone\b", r"\bmeganz\b", r"\bwinscp\b",
    r"\bexfil\w*\b", r"\bdata\s+theft\b", r"\bdouble\s+extortion\b", r"\bleak\s+site\b",
    r"\bshadow\s+copy\b", r"\bvss\b", r"\bransom\s+note\b", r"\bencrypt\w*\b",
    r"\bscheduled\s+task\b", r"\bgpo\b", r"\bgroup\s+policy\b",
]

DFIR_CASE_STUDY_SIGNALS = [
    r"\bcase\s+summary\b", r"\bintrusion\b", r"\bwe\s+observed\b", r"\btimeline\b",
    r"\bbeach\s*head\b", r"\bday\s+\d+\b", r"\bincident\s+response\b", r"\bforensic\w*\b",
]

PLATFORM_PATTERNS = {
    "ESXi": [r"\besxi\b", r"\bvmware\b", r"\bvcenter\b"],
    "Linux": [r"\blinux\b", r"\bubuntu\b", r"\bdebian\b", r"\bcentos\b", r"\brhel\b", r"\bred\s*hat\b"],
    "Windows": [r"\bwindows\b", r"\bactive\s+directory\b", r"\bdomain\s+controller\b", r"\blsass\b"],
}

DEPTH_HIGH = [
    r"\batt&ck\b", r"\bmitre\b", r"\btelemetry\b", r"\bioc(s)?\b", r"\byara\b", r"\bsigma\b",
    r"\bpcap\b", r"\bmemory\s+dump\b", r"\breverse\s+engineering\b", r"\bprocess\s+injection\b",
    r"\blog(s)?\b", r"\bevidence\b", r"\bdetection\s+rule\b",
]
DEPTH_MED = [
    r"\bkill\s+chain\b", r"\blateral\s+movement\b", r"\bprivilege\s+escalation\b",
    r"\bexfiltration\b", r"\bpersistence\b", r"\bdiscovery\b",
]

RED_FLAG_PATTERNS = {
    "Marketing": [
        r"\b(contact\s+us|pricing|book\s+a\s+demo|request\s+a\s+demo)\b",
        r"\bour\s+product\b", r"\bwebinar\b", r"\bsponsored\b",
    ],
    "News": [
        r"\bbreaking\s+news\b", r"\bheadlines\b", r"\bnews\s+update\b",
    ],
    "LowSignal": [
        r"\b(what\s+is\s+ransomware|ransomware\s+explained)\b"
    ]
}

def normalize(text: str) -> str:
    return re.sub(r"\s+", " ", text.lower())

def any_hit(text: str, patterns):
    return any(re.search(p, text, flags=re.IGNORECASE) for p in patterns)

def count_hits(text: str, patterns):
    return sum(len(re.findall(p, text, flags=re.IGNORECASE)) for p in patterns)

def detect_platform(text: str) -> str:
    hits = {plat: count_hits(text, pats) for plat, pats in PLATFORM_PATTERNS.items()}
    chosen = [p for p, h in hits.items() if h > 0]
    if not chosen:
        return ""
    if len(chosen) == 1:
        return chosen[0]
    return "Mixed"

def detect_depth(text: str) -> str:
    high = count_hits(text, DEPTH_HIGH)
    med = count_hits(text, DEPTH_MED)
    if high >= 6:
        return "High"
    if high >= 2 or med >= 4:
        return "Medium"
    return "Low"

def detect_red_flags(text: str) -> str:
    flags = [label for label, pats in RED_FLAG_PATTERNS.items() if any_hit(text, pats)]
    return "/".join(flags)

def recency_bucket(published_at: str) -> str:
    s = str(published_at).strip()
    if not s:
        return ""
    m = re.search(r"(\d{4})", s)
    if not m:
        return ""
    y = int(m.group(1))
    if y < 2020:
        return "Pre-2020"
    if 2020 <= y <= 2022:
        return "2020-2022"
    return "2023+"

# -----------------------
# Family aliases → detect specific family
# -----------------------
def load_family_aliases_xlsx(path: Path):
    df = pd.read_excel(path).fillna("")
    alias_map = {}
    for _, row in df.iterrows():
        fam = str(row.get("Ransomware_Family_Name","")).strip()
        if not fam:
            continue
        alias_map[fam.lower()] = fam
        aliases = str(row.get("Alias_Names","")).strip()
        if aliases and aliases != "—":
            for a in [x.strip() for x in aliases.split(",")]:
                if a:
                    alias_map[a.lower()] = fam
    return alias_map

def find_families(text: str, alias_map):
    """Return a sorted list of matched canonical families."""
    matched = set()
    for alias, canon in alias_map.items():
        alias_re = re.escape(alias).replace(r"\ ", r"\s+")
        if re.search(rf"\b{alias_re}\b", text, flags=re.IGNORECASE):
            matched.add(canon)
    return sorted(matched)

alias_map = load_family_aliases_xlsx(FAMILY_LIST)

# -----------------------
# Index transcripts recursively
# -----------------------
txt_files = sorted(TRANSCRIPTS_ROOT.rglob("*.txt"))
txt_index = {p.stem: p for p in txt_files}

meta_index = {}
for p in TRANSCRIPTS_ROOT.rglob("*.meta.json"):
    vid = p.name.replace(".meta.json", "")
    meta_index[vid] = p

print("Indexed transcripts:", len(txt_index))
print("Indexed meta json:", len(meta_index))

# -----------------------
# Load rubric WITH formulas and fill only target columns
# -----------------------
wb = load_workbook(RUBRIC_IN)
ws = wb["Video_Selection_Rubric"] if "Video_Selection_Rubric" in wb.sheetnames else wb.active

# header map
headers = {}
for c in range(1, ws.max_column + 1):
    v = ws.cell(row=1, column=c).value
    if isinstance(v, str) and v.strip():
        headers[v.strip()] = c

def h(name: str):
    if name not in headers:
        raise KeyError(f"Missing column in rubric template: {name}")
    return headers[name]

# columns we will fill (leave formulas alone)
COL_VIDEO_ID = h("Video_ID")
COL_CH_ID = headers.get("Channel_ID")
COL_CH_NAME = headers.get("Channel_Name")
COL_TITLE = headers.get("Video_Title")
COL_RANSOM = h("Ransomware_Family_Mentioned (Yes/No)")
COL_SPEC = h("Specific_Family_Named (Yes/No)")
COL_TTPS = h("TTPs_Mentioned (Yes/No)")
COL_DFIR = h("DFIR_Case_Study (Yes/No)")
COL_PLATFORM = h("Platform_Mentioned (Windows/Linux/ESXi)")
COL_TRANSCRIPT = h("Transcript_Available (Yes/No)")
COL_MINUTES = headers.get("Video_Length_Minutes")
COL_DEPTH = h("Estimated_Technical_Depth (Low/Medium/High)")
COL_RECENCY = h("Recency (Pre-2020 / 2020-2022 / 2023+)")
COL_REDFLAG = h("Red_Flags (News/Marketing/LowSignal)")
COL_NOTES = headers.get("Notes")

def set_val(row, col, val):
    if col is None:
        return
    ws.cell(row=row, column=col, value=val)

filled = 0
missing = 0

for r in range(3, ws.max_row + 1):
    vid = ws.cell(row=r, column=COL_VIDEO_ID).value
    if not vid:
        continue
    vid = str(vid).strip()
    if not vid.startswith("V"):
        continue

    tfile = txt_index.get(vid)
    if not tfile:
        set_val(r, COL_TRANSCRIPT, "No")
        missing += 1
        continue

    text_raw = tfile.read_text(encoding="utf-8", errors="ignore")
    text = normalize(text_raw)

    meta = {}
    mp = meta_index.get(vid)
    if mp and mp.exists():
        meta = json.loads(mp.read_text(encoding="utf-8"))

    # Meta columns
    set_val(r, COL_CH_ID, meta.get("Channel_ID",""))
    set_val(r, COL_CH_NAME, meta.get("Channel_Name",""))
    set_val(r, COL_TITLE, meta.get("Video_Title",""))

    # Transcript exists -> available
    set_val(r, COL_TRANSCRIPT, "Yes")

    # Minutes + recency from meta
    if COL_MINUTES is not None:
        dur = meta.get("DurationSeconds", "")
        try:
            mins = round(float(dur)/60.0, 2) if dur != "" else ""
        except Exception:
            mins = ""
        set_val(r, COL_MINUTES, mins)

    bucket = recency_bucket(meta.get("PublishedAt",""))
    if bucket:
        set_val(r, COL_RECENCY, bucket)

    # Transcript-derived rubric signals
    set_val(r, COL_RANSOM, "Yes" if re.search(r"\bransomware\b", text, re.I) else "No")

    fams = find_families(text, alias_map)
    set_val(r, COL_SPEC, "Yes" if len(fams) > 0 else "No")

    set_val(r, COL_TTPS, "Yes" if any_hit(text, TTP_KEYWORDS) else "No")
    set_val(r, COL_DFIR, "Yes" if any_hit(text, DFIR_CASE_STUDY_SIGNALS) else "No")

    plat = detect_platform(text)
    if plat:
        set_val(r, COL_PLATFORM, plat)
    else:
        set_val(r, COL_PLATFORM, "")

    set_val(r, COL_DEPTH, detect_depth(text))

    flags = detect_red_flags(text)
    set_val(r, COL_REDFLAG, flags)

    # Notes: write matched families (optional but nice for “clear picture”)
    if COL_NOTES is not None and fams:
        set_val(r, COL_NOTES, "Families: " + ", ".join(fams[:6]))

    filled += 1

wb.save(RUBRIC_OUT)
print("Rows filled:", filled, "| missing transcripts:", missing)
print("Saved:", RUBRIC_OUT)

Indexed transcripts: 440
Indexed meta json: 440
Rows filled: 439 | missing transcripts: 0
Saved: /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/outputs/Video_Selection_Rubric_Filled_Columns_ONLY.xlsx


In [5]:
import pandas as pd
from pathlib import Path
import numpy as np

BASE = Path("/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent")

rubric_filled = BASE / "outputs/Video_Selection_Rubric_Filled_Columns_ONLY.xlsx"
df = pd.read_excel(rubric_filled).fillna("")

def yn(x):
    return str(x).strip().lower() == "yes"

def depth_score(x):
    x = str(x).strip().lower()
    return 2 if x == "high" else 1 if x == "medium" else 0

def recency_score(x):
    x = str(x).strip()
    return 1 if x == "2023+" else 0.5 if x == "2020-2022" else 0

def platform_score(x):
    return 1 if str(x).strip() != "" else 0

def redflag_penalty(x):
    return -2 if str(x).strip() != "" else 0

# Drop guidance/non-video rows safely
df = df[df["Video_ID"].astype(str).str.match(r"^V\d+", na=False)].copy()

df["Selection_Score_PY"] = (
    df["Ransomware_Family_Mentioned (Yes/No)"].apply(lambda v: 1 if yn(v) else 0) +
    df["Specific_Family_Named (Yes/No)"].apply(lambda v: 2 if yn(v) else 0) +
    df["TTPs_Mentioned (Yes/No)"].apply(lambda v: 2 if yn(v) else 0) +
    df["DFIR_Case_Study (Yes/No)"].apply(lambda v: 2 if yn(v) else 0) +
    df["Platform_Mentioned (Windows/Linux/ESXi)"].apply(platform_score) +
    df["Transcript_Available (Yes/No)"].apply(lambda v: 2 if yn(v) else 0) +
    df["Estimated_Technical_Depth (Low/Medium/High)"].apply(depth_score) +
    df["Recency (Pre-2020 / 2020-2022 / 2023+)"].apply(recency_score) +
    df["Red_Flags (News/Marketing/LowSignal)"].apply(redflag_penalty)
)

df["Include_PY"] = df["Selection_Score_PY"].apply(lambda s: "Yes" if s >= 6 else "No")

print("Total videos:", len(df))
print("Included (Yes):", (df["Include_PY"]=="Yes").sum())
df[["Video_ID","Selection_Score_PY","Include_PY"]].head(10)

Total videos: 440
Included (Yes): 307


,Video_ID,Selection_Score_PY,Include_PY
0,V0210,2.0,No
1,V0205,7.5,Yes
2,V0212,3.5,No
3,V0203,7.5,Yes
4,V0204,9.5,Yes
5,V0211,7.5,Yes
6,V0207,4.5,No
7,V0208,4.5,No
8,V0206,7.5,Yes
9,V0209,8.5,Yes


In [6]:
out_scored = BASE / "outputs/Video_Selection_Rubric_FINAL_Scored.xlsx"
df.to_excel(out_scored, index=False)
print("Saved scored rubric:", out_scored)

selected = df[df["Include_PY"] == "Yes"].copy()

# Create a clean Video Registry dataset (you can add/remove columns)
video_registry = selected[[
    "Video_ID",
    "Channel_ID",
    "Channel_Name",
    "Video_Title",
    "Video_Length_Minutes",
    "PublishedAt",
    "Recency (Pre-2020 / 2020-2022 / 2023+)",
    "Platform_Mentioned (Windows/Linux/ESXi)",
    "Estimated_Technical_Depth (Low/Medium/High)",
    "Ransomware_Family_Mentioned (Yes/No)",
    "Specific_Family_Named (Yes/No)",
    "TTPs_Mentioned (Yes/No)",
    "DFIR_Case_Study (Yes/No)",
    "Red_Flags (News/Marketing/LowSignal)",
    "Selection_Score_PY",
    "Include_PY",
    "Notes"
]].copy()

out_registry = BASE / "outputs/Video_Registry_Dataset.xlsx"
video_registry.to_excel(out_registry, index=False)

print("Saved Video Registry dataset:", out_registry)
print("Selected videos:", len(video_registry))

Saved scored rubric: /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/outputs/Video_Selection_Rubric_FINAL_Scored.xlsx
Saved Video Registry dataset: /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/outputs/Video_Registry_Dataset.xlsx
Selected videos: 307


In [7]:
import pandas as pd
import json
from pathlib import Path

BASE = Path("/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent")

IN_XLSX  = BASE / "outputs/Video_Registry_Dataset.xlsx"
OUT_XLSX = BASE / "outputs/Video_Registry_Dataset_with_URL_and_Path.xlsx"
TRANSCRIPTS_ROOT = BASE / "transcripts"

# Load registry
df = pd.read_excel(IN_XLSX).fillna("")

# Build indices from your transcript folders (recursive)
txt_index = {p.stem: p for p in TRANSCRIPTS_ROOT.rglob("*.txt")}

meta_index = {}
for p in TRANSCRIPTS_ROOT.rglob("*.meta.json"):
    vid = p.name.replace(".meta.json", "")  # V0001.meta.json -> V0001
    meta_index[vid] = p

youtube_urls = []
transcript_paths = []
youtube_ids = []

for vid in df["Video_ID"].astype(str).str.strip():
    # Transcript path
    tpath = txt_index.get(vid)
    transcript_paths.append(str(tpath) if tpath else "")

    # YouTube ID from meta json (if present)
    yid = ""
    mp = meta_index.get(vid)
    if mp and mp.exists():
        try:
            meta = json.loads(mp.read_text(encoding="utf-8"))
            yid = meta.get("YouTube_Video_ID", "") or meta.get("youtube_video_id", "") or meta.get("video_id", "")
        except Exception:
            yid = ""
    youtube_ids.append(yid)
    youtube_urls.append(f"https://www.youtube.com/watch?v={yid}" if yid else "")

# Add/overwrite columns
df["Transcript_Path"] = transcript_paths
df["YouTube_URL"] = youtube_urls

# Optional but helpful: keep the ID too (in case you want to sort/check missing URLs)
if "YouTube_Video_ID" not in df.columns:
    df["YouTube_Video_ID"] = youtube_ids

# Save
df.to_excel(OUT_XLSX, index=False)

print("Saved:", OUT_XLSX)
print("Missing Transcript_Path:", (df["Transcript_Path"] == "").sum())
print("Missing YouTube_URL:", (df["YouTube_URL"] == "").sum())
df[["Video_ID", "YouTube_URL", "Transcript_Path"]].head(10)

Saved: /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/outputs/Video_Registry_Dataset_with_URL_and_Path.xlsx
Missing Transcript_Path: 0
Missing YouTube_URL: 0


,Video_ID,YouTube_URL,Transcript_Path
0,V0205,https://www.youtube.com/watch?v=gtLXRQFCgOg,/home/henrykabuye/Ransap/Publications 2026/Pro...
1,V0203,https://www.youtube.com/watch?v=H7b704TFhW4,/home/henrykabuye/Ransap/Publications 2026/Pro...
2,V0204,https://www.youtube.com/watch?v=VQO8HQSA54I,/home/henrykabuye/Ransap/Publications 2026/Pro...
3,V0211,https://www.youtube.com/watch?v=RsMSnVUsNpQ,/home/henrykabuye/Ransap/Publications 2026/Pro...
4,V0206,https://www.youtube.com/watch?v=pPyTJO5c4_U,/home/henrykabuye/Ransap/Publications 2026/Pro...
5,V0209,https://www.youtube.com/watch?v=M85zrOA59OE,/home/henrykabuye/Ransap/Publications 2026/Pro...
6,V0183,https://www.youtube.com/watch?v=YQEq5s4SRxY,/home/henrykabuye/Ransap/Publications 2026/Pro...
7,V0190,https://www.youtube.com/watch?v=duLJUpptSik,/home/henrykabuye/Ransap/Publications 2026/Pro...
8,V0188,https://www.youtube.com/watch?v=LxUAnZY_08o,/home/henrykabuye/Ransap/Publications 2026/Pro...
9,V0191,https://www.youtube.com/watch?v=nxlm7pIvMdg,/home/henrykabuye/Ransap/Publications 2026/Pro...


In [8]:
# Confirming .txt and .meta.json pairing

import pandas as pd
from pathlib import Path

BASE = Path("/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent")
TRANSCRIPTS_ROOT = BASE / "transcripts"
REGISTRY = BASE / "outputs/Video_Registry_Dataset_with_URL_and_Path.xlsx"

df = pd.read_excel(REGISTRY).fillna("")

missing_txt = []
missing_meta = []

for _, row in df.iterrows():
    tpath = Path(str(row["Transcript_Path"]))
    if not tpath.exists():
        missing_txt.append(row["Video_ID"])
        continue

    meta_path = tpath.with_suffix(".meta.json")
    if not meta_path.exists():
        missing_meta.append(row["Video_ID"])

print("Missing .txt:", len(missing_txt))
print("Missing .meta.json:", len(missing_meta))

# Show a few examples if any
print("Examples missing txt:", missing_txt[:5])
print("Examples missing meta:", missing_meta[:5])

Missing .txt: 0
Missing .meta.json: 0
Examples missing txt: []
Examples missing meta: []


In [9]:
import pandas as pd
from pathlib import Path

BASE = Path("/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent")
registry = BASE / "outputs/Video_Registry_Dataset_with_URL_and_Path.xlsx"

df = pd.read_excel(registry)

print("Total selected videos:", len(df))

print("\nVideos < 5 minutes:")
print(df[df["Video_Length_Minutes"] < 5][["Video_ID","Video_Length_Minutes"]].head())

print("\nCount < 5 minutes:", (df["Video_Length_Minutes"] < 5).sum())

Total selected videos: 307

Videos < 5 minutes:
Empty DataFrame
Columns: [Video_ID, Video_Length_Minutes]
Index: []

Count < 5 minutes: 0


In [10]:
import pandas as pd
import re
from pathlib import Path

BASE = Path("/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent")
REGISTRY = BASE / "outputs/Video_Registry_Dataset_with_URL_and_Path.xlsx"
FAMILY_LIST = BASE / "rubrics/Ransomware_Family_Coverage_List.xlsx"

df = pd.read_excel(REGISTRY).fillna("")

# -------------------------
# Load family aliases
# -------------------------
fam_df = pd.read_excel(FAMILY_LIST).fillna("")
alias_to_family = {}

for _, row in fam_df.iterrows():
    fam = str(row.get("Ransomware_Family_Name", "")).strip()
    if not fam:
        continue
    alias_to_family[fam.lower()] = fam
    aliases = str(row.get("Alias_Names", "")).strip()
    if aliases and aliases != "—":
        for a in [x.strip() for x in aliases.split(",")]:
            if a:
                alias_to_family[a.lower()] = fam

alias_items = list(alias_to_family.items())

def normalize(text):
    return re.sub(r"\s+", " ", text.lower())

def match_aliases(text):
    matched = set()
    for alias, fam in alias_items:
        alias_re = re.escape(alias).replace(r"\ ", r"\s+")
        if re.search(rf"\b{alias_re}\b", text, flags=re.IGNORECASE):
            matched.add(fam)
    return sorted(matched)

# -------------------------
# ATT&CK-lite tactic patterns (editable)
# -------------------------
TACTICS = {
    "Initial_Access": [
        r"\bphish", r"\brdp\b", r"\bexploit", r"\bvulnerab", r"\bdrive[- ]by\b",
        r"\bmalspam\b", r"\bcredential stuffing\b", r"\bpassword spray\b"
    ],
    "Execution": [
        r"\bpowershell\b", r"\bcmd\.exe\b", r"\bscript\b", r"\bhta\b", r"\bmacro\b"
    ],
    "Persistence": [
        r"\bscheduled task\b", r"\brun key\b", r"\bservice\b", r"\bstartup\b"
    ],
    "Privilege_Escalation": [
        r"\bprivilege escalation\b", r"\buac\b", r"\btoken\b"
    ],
    "Credential_Access": [
        r"\bmimikatz\b", r"\blsass\b", r"\bcredential dump\b", r"\bntlm\b"
    ],
    "Lateral_Movement": [
        r"\bpsexec\b", r"\bwmi\b", r"\bwinrm\b", r"\brdp\b", r"\bpass[- ]the[- ]hash\b"
    ],
    "Discovery": [
        r"\bnetwork scan\b", r"\bad enumeration\b", r"\bbloodhound\b"
    ],
    "Command_and_Control": [
        r"\bcobalt strike\b", r"\bbeacon\b", r"\breverse shell\b"
    ],
    "Exfiltration": [
        r"\bexfil", r"\brclone\b", r"\bmeganz\b", r"\bdouble extortion\b", r"\bleak site\b"
    ],
    "Impact": [
        r"\bencrypt", r"\bransom note\b", r"\bshadow copy\b", r"\bvss\b", r"\bdata destruction\b"
    ]
}

TOOLS = {
    "Cobalt_Strike": [r"\bcobalt strike\b", r"\bbeacon\b"],
    "Mimikatz": [r"\bmimikatz\b"],
    "PsExec": [r"\bpsexec\b"],
    "Rclone": [r"\brclone\b"],
    "MegaNZ": [r"\bmega\.?nz\b", r"\bmeganz\b"],
    "AnyDesk": [r"\banydesk\b"],
    "TeamViewer": [r"\bteamviewer\b"],
    "BloodHound": [r"\bbloodhound\b"],
}

PLATFORMS = {
    "Windows": [r"\bwindows\b", r"\bactive directory\b", r"\bdomain controller\b", r"\blsass\b"],
    "Linux": [r"\blinux\b", r"\bubuntu\b", r"\bdebian\b", r"\bcentos\b", r"\brhel\b"],
    "ESXi": [r"\besxi\b", r"\bvmware\b", r"\bvcenter\b"]
}

def count_hits(text, patterns):
    return sum(len(re.findall(p, text, flags=re.IGNORECASE)) for p in patterns)

records = []

for _, row in df.iterrows():
    vid = row["Video_ID"]
    tpath = Path(row["Transcript_Path"])
    text = normalize(tpath.read_text(encoding="utf-8", errors="ignore"))

    families = match_aliases(text)

    tactic_counts = {t: count_hits(text, pats) for t, pats in TACTICS.items()}
    tool_counts = {k: count_hits(text, pats) for k, pats in TOOLS.items()}
    platform_counts = {p: count_hits(text, pats) for p, pats in PLATFORMS.items()}

    records.append({
        "Video_ID": vid,
        "Channel_ID": row.get("Channel_ID",""),
        "Channel_Name": row.get("Channel_Name",""),
        "Video_Title": row.get("Video_Title",""),
        "YouTube_URL": row.get("YouTube_URL",""),
        "Transcript_Path": str(tpath),

        "Family_Count": len(families),
        "Family_List": ", ".join(families),

        "Tool_Total_Mentions": sum(tool_counts.values()),
        "Tool_List": ", ".join([k for k,v in tool_counts.items() if v > 0]),

        "Tactic_Total_Mentions": sum(tactic_counts.values()),
        "Dominant_Tactic": max(tactic_counts, key=tactic_counts.get) if sum(tactic_counts.values())>0 else "",

        "Platform_Signal": max(platform_counts, key=platform_counts.get) if sum(platform_counts.values())>0 else "",
        **{f"Tactic_{k}": v for k,v in tactic_counts.items()},
        **{f"Tool_{k}": v for k,v in tool_counts.items()},
        **{f"Platform_{k}": v for k,v in platform_counts.items()},
    })

feat = pd.DataFrame(records)

out_xlsx = BASE / "outputs/Knowledge_Agent_Features_307.xlsx"
out_csv  = BASE / "outputs/Knowledge_Agent_Features_307.csv"

feat.to_excel(out_xlsx, index=False)
feat.to_csv(out_csv, index=False)

print("Saved:", out_xlsx)
print("Saved:", out_csv)
feat.head()

Saved: /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/outputs/Knowledge_Agent_Features_307.xlsx
Saved: /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/outputs/Knowledge_Agent_Features_307.csv


,Video_ID,Channel_ID,Channel_Name,Video_Title,YouTube_URL,Transcript_Path,Family_Count,Family_List,Tool_Total_Mentions,Tool_List,...,Tool_Mimikatz,Tool_PsExec,Tool_Rclone,Tool_MegaNZ,Tool_AnyDesk,Tool_TeamViewer,Tool_BloodHound,Platform_Windows,Platform_Linux,Platform_ESXi
0,V0205,C22,Threatpost,Patrick Wardle: Hackers Abusing Apple Technolo...,https://www.youtube.com/watch?v=gtLXRQFCgOg,/home/henrykabuye/Ransap/Publications 2026/Pro...,1,Play,0,,...,0,0,0,0,0,0,0,6,1,0
1,V0203,C22,Threatpost,Malware Partnerships: A Formidable Threat to B...,https://www.youtube.com/watch?v=H7b704TFhW4,/home/henrykabuye/Ransap/Publications 2026/Pro...,2,"Egregor, Ryuk",0,,...,0,0,0,0,0,0,0,0,0,0
2,V0204,C22,Threatpost,"Email Security Attacks End in Financial Ruin, ...",https://www.youtube.com/watch?v=VQO8HQSA54I,/home/henrykabuye/Ransap/Publications 2026/Pro...,1,Play,0,,...,0,0,0,0,0,0,0,0,0,0
3,V0211,C22,Threatpost,"Amid Olympics Uncertainty, 5G Security Lessons...",https://www.youtube.com/watch?v=RsMSnVUsNpQ,/home/henrykabuye/Ransap/Publications 2026/Pro...,2,"Play, Qilin",0,,...,0,0,0,0,0,0,0,1,0,0
4,V0206,C22,Threatpost,Ransomware And IP Theft Top COVID-19 Healthcar...,https://www.youtube.com/watch?v=pPyTJO5c4_U,/home/henrykabuye/Ransap/Publications 2026/Pro...,0,,0,,...,0,0,0,0,0,0,0,0,0,0


In [1]:
import pandas as pd

df = pd.read_csv("/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/outputs/Knowledge_Agent_Features_307.csv")

print("\nTop ransomware families:")
print(df["Family_List"].str.split(", ").explode().value_counts().head(15))

print("\nMost mentioned tactics:")
print(df.filter(like="Tactic_").sum().sort_values(ascending=False).head(10))

print("\nMost mentioned tools:")
print(df.filter(like="Tool_").sum().sort_values(ascending=False).head(10))

print("\nPlatform distribution:")
print(df.filter(like="Platform_").sum())


Top ransomware families:
Family_List
Play        149
Qilin        37
Akira        10
Royal         9
LockBit       9
Hive          7
INC           5
Conti         3
Babuk         2
Medusa        2
Egregor       1
Ryuk          1
Cl0p          1
DarkSide      1
BlackCat      1
Name: count, dtype: int64

Most mentioned tactics:
Tactic_Total_Mentions          5043
Tactic_Initial_Access          2105
Tactic_Execution                791
Tactic_Persistence              772
Tactic_Impact                   651
Tactic_Privilege_Escalation     237
Tactic_Exfiltration             209
Tactic_Lateral_Movement         160
Tactic_Command_and_Control      111
Tactic_Discovery                  4
dtype: int64

Most mentioned tools:


TypeError: unsupported operand type(s) for +: 'int' and 'str'

In [2]:
tool_df = df.filter(like="Tool_").select_dtypes(include="number")

print("\nMost mentioned tools:")
print(tool_df.sum().sort_values(ascending=False).head(10))


Most mentioned tools:
Tool_Total_Mentions    130
Tool_Cobalt_Strike      85
Tool_AnyDesk            28
Tool_Rclone             14
Tool_BloodHound          3
Tool_Mimikatz            0
Tool_PsExec              0
Tool_MegaNZ              0
Tool_TeamViewer          0
dtype: int64


In [3]:
import pandas as pd

df = pd.read_csv("/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/outputs/Knowledge_Agent_Features_307.csv")

print("\nTop ransomware families:")
print(df["Family_List"].str.split(", ").explode().value_counts().head(15))

print("\nMost mentioned tactics:")
print(df.filter(like="Tactic_").select_dtypes(include="number").sum().sort_values(ascending=False).head(10))

print("\nMost mentioned tools:")
print(df.filter(like="Tool_").select_dtypes(include="number").sum().sort_values(ascending=False).head(10))

print("\nPlatform distribution:")
print(df.filter(like="Platform_").select_dtypes(include="number").sum().sort_values(ascending=False))


Top ransomware families:
Family_List
Play        149
Qilin        37
Akira        10
Royal         9
LockBit       9
Hive          7
INC           5
Conti         3
Babuk         2
Medusa        2
Egregor       1
Ryuk          1
Cl0p          1
DarkSide      1
BlackCat      1
Name: count, dtype: int64

Most mentioned tactics:
Tactic_Total_Mentions          5043
Tactic_Initial_Access          2105
Tactic_Execution                791
Tactic_Persistence              772
Tactic_Impact                   651
Tactic_Privilege_Escalation     237
Tactic_Exfiltration             209
Tactic_Lateral_Movement         160
Tactic_Command_and_Control      111
Tactic_Discovery                  4
dtype: int64

Most mentioned tools:
Tool_Total_Mentions    130
Tool_Cobalt_Strike      85
Tool_AnyDesk            28
Tool_Rclone             14
Tool_BloodHound          3
Tool_Mimikatz            0
Tool_PsExec              0
Tool_MegaNZ              0
Tool_TeamViewer          0
dtype: int64

Platform distribut

In [5]:
import shutil
shutil.copy(
"/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/outputs/Knowledge_Agent_Features_307.csv",
"/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/datasets/Knowledge_Agent_Features_v1.csv"
)

'/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/datasets/Knowledge_Agent_Features_v1.csv'